# Week 06 - Day 04: Spotify Snowflake Exercises
**Dataset:** `Spotify data.csv` (2,017 songs, 17 columns)

### Setup: Create Database, Schema, Warehouse, Table & Load CSV

In [1]:
-- 1. Create Database, Schema, and Warehouse
CREATE DATABASE IF NOT EXISTS spotify_db;
CREATE SCHEMA IF NOT EXISTS spotify_db.music;

CREATE WAREHOUSE IF NOT EXISTS spotify_wh
  WAREHOUSE_SIZE = 'XSMALL'
  AUTO_SUSPEND = 60
  AUTO_RESUME = TRUE;

USE WAREHOUSE spotify_wh;
USE SCHEMA spotify_db.music;

-- 2. Create the CSV File Format
CREATE OR REPLACE FILE FORMAT spotify_csv_ff
  TYPE = CSV
  SKIP_HEADER = 1
  FIELD_OPTIONALLY_ENCLOSED_BY = '"'
  ESCAPE_UNENCLOSED_FIELD = NONE
  NULL_IF = ('');

-- 3. Create the spotify_songs Table (17 columns matching the CSV)
CREATE OR REPLACE TABLE spotify_songs (
  song_id INT,
  acousticness FLOAT,
  danceability FLOAT,
  duration_ms INT,
  energy FLOAT,
  instrumentalness FLOAT,
  key INT,
  liveness FLOAT,
  loudness FLOAT,
  mode INT,
  speechiness FLOAT,
  tempo FLOAT,
  time_signature INT,
  valence FLOAT,
  target INT,
  song_title STRING,
  artist STRING
);

-- 4. Create the Stage to upload the CSV file
CREATE OR REPLACE STAGE spotify_stage FILE_FORMAT = spotify_csv_ff;

-- 5. Copy the uploaded CSV into the spotify_songs table
COPY INTO spotify_songs FROM @spotify_stage ON_ERROR = 'CONTINUE';

FILE,STATUS,ROWS_PARSED,ROWS_LOADED
spotify_stage/Spotify data.csv,LOADED,2017,2017


### Q1. Which is the longest song released yet?

In [2]:
SELECT song_title, artist, duration_ms, ROUND(duration_ms / 60000, 2) AS duration_minutes
FROM spotify_songs
ORDER BY duration_ms DESC
LIMIT 1;

SONG_TITLE,ARTIST,DURATION_MS,DURATION_MINUTES
The Nearness of You,Joshua Redman,1004627,16.74


### Q2. How many songs has Drake sung?

In [3]:
SELECT COUNT(*) AS drake_songs
FROM spotify_songs
WHERE artist = 'Drake';

DRAKE_SONGS
16


### Q3. List those songs with 'Love' in them.

In [4]:
SELECT song_title, artist
FROM spotify_songs
WHERE song_title ILIKE '%Love%';

SONG_TITLE,ARTIST
House of Jealous Lovers,The Rapture
If I Gave You My Love,Myron & E
Who Be Lovin Me (feat. ILOVEMAKONNEN),Santigold
Ain't No Love In The Heart Of The City - Single Version,"Bobby ""Blue"" Bland"
Our Love,Caribou
Have Love Will Travel,The Sonics
I'm In Love,Fool's Gold
Why Don't They Let Us Fall in Love,The Ronettes
Will You Love Me Tomorrow,The Shirelles
Let Me Show You Love,Cut Copy


### Q4. Select songs which have a number in them.

In [5]:
SELECT song_title, artist
FROM spotify_songs
WHERE REGEXP_LIKE(song_title, '.*[0-9].*');

SONG_TITLE,ARTIST
07-Nov,Beyoncé
He's So Fine - 2000 Digital Remaster,The Chiffons
Queen's Speech 4,Lady Leshurr
679 (feat. Remy Boyz),Fetty Wap
My Sub (Pt. 2: The Jackin') - Album Version (Edited),Big K.R.I.T.
9,Willow
Hercules Theme 2014,Hercules & Love Affair
Fame (2007 Remastered),David Bowie
Every 1's A Winner - Single Version,Hot Chocolate
I Would Die 4 U,Prince


### Q5. You have a bachelorette weekend, which songs would you play the most?

In [6]:
SELECT song_title, artist, danceability, energy, valence
FROM spotify_songs
WHERE danceability > 0.8
  AND energy > 0.8
  AND valence > 0.75
ORDER BY danceability DESC, energy DESC
LIMIT 10;

SONG_TITLE,ARTIST,DANCEABILITY,ENERGY,VALENCE
Fuck Kd,StaRR Lyfe,0.947,0.829,0.837
Sippin On Some Syrup,Three 6 Mafia,0.944,0.839,0.891
Hollaback Girl,Gwen Stefani,0.926,0.916,0.903
Hoochie Mama,2 LIVE CREW,0.895,0.883,0.893
Candy,Dillon Francis,0.894,0.892,0.867
My Favorite Snack,Imagination Movers,0.890,0.891,0.947
Sigo Extrañándote,J Balvin,0.889,0.853,0.916
P.Y.T. (Pretty Young Thing),Michael Jackson,0.888,0.815,0.960
Bustin' Loose,Chuck Brown and the Soul Searchers,0.865,0.896,0.939
Piss Test (feat. Juicy J & Dany Brown),A-Trak,0.864,0.860,0.883


### Q6. In a dance battle, which 3 songs would you choose for a dance off?

In [7]:
SELECT song_title, artist, danceability, energy, tempo
FROM spotify_songs
ORDER BY danceability DESC, energy DESC, tempo DESC
LIMIT 3;

SONG_TITLE,ARTIST,DANCEABILITY,ENERGY,TEMPO
Flashwind - Radio Edit,Ben Remember,0.984,0.941,122.003
SexyBack,Justin Timberlake,0.967,0.583,117.000
Check Me Out Like,Blaqstarr,0.962,0.592,130.684


### Q7. Which artist has the max songs released?

In [8]:
SELECT artist, COUNT(*) AS total_songs
FROM spotify_songs
GROUP BY artist
ORDER BY total_songs DESC
LIMIT 1;

ARTIST,TOTAL_SONGS
Drake,16


### Q8. Make a 'break-up' song list >..<

In [9]:
SELECT song_title, artist, valence, energy, acousticness
FROM spotify_songs
WHERE valence < 0.25
  AND energy < 0.4
  AND acousticness > 0.7
  AND instrumentalness < 0.5
ORDER BY valence ASC
LIMIT 10;

SONG_TITLE,ARTIST,VALENCE,ENERGY,ACOUSTICNESS
"My Heart Will Go On (Love Theme from ""Titanic"")",Céline Dion,0.0384,0.3640,0.707
"Trio Sonata in G Major, Wq. 144: I. Adagio",Carl Philipp Emanuel Bach,0.0484,0.0626,0.913
"Trio No. 2 en Mi Mineur, Op. 92: III. Andante con moto",Camille Saint-Saëns,0.0559,0.0632,0.983
Fool,Perfume Genius,0.0576,0.2900,0.712
"Piano Quartet in E flat, Op.47: 3. Andante cantabile",Robert Schumann,0.0588,0.0230,0.953
Where We Used To Live,Esbjörn Svensson Trio,0.0718,0.0867,0.871
"String Quartet No. 4 in C Major, D. 46: II. Andante con moto",Franz Schubert,0.0855,0.0156,0.915
Stay,Rihanna,0.0968,0.3240,0.949
The Nearness Of You,Bill Charlap,0.0999,0.1310,0.761
Gooey,Glass Animals,0.1070,0.3760,0.726
